In [30]:
from dotenv import load_dotenv
from os import getenv
import vk_utils


import importlib
importlib.reload(vk_utils)


load_dotenv()


new_communities = vk_utils.get_new_communities(api_key=getenv(
    "VK_API_KEY"), queries=["подслушано", "район"], state_dsn=getenv("POSTGRESQL_DSN"))
print("Найдено сообществ:", len(new_communities))
new_communities[:10]

Найдено сообществ: 187


[{'id': 224949751,
  'name': 'ПОДСЛУШАНО НА СВО / Выплаты, Поиск пропавших',
  'category': 'Дискуссионный клуб'},
 {'id': 185347562,
  'name': 'Подслушано ФСИН России',
  'category': 'Группа коллег'},
 {'id': 99484444,
  'name': 'Подслушано Краснодар | Типичный',
  'category': 'Городское сообщество'},
 {'id': 65140014,
  'name': 'Подслушано Родники',
  'category': 'Городское сообщество'},
 {'id': 161280734,
  'name': 'Подслушано в Назарово',
  'category': 'Городское сообщество'},
 {'id': 143277166,
  'name': 'ПОДСЛУШАНО НА СВО',
  'category': 'Дискуссионный клуб'},
 {'id': 189070626,
  'name': 'Подслушано Беременна в 16',
  'category': 'Философия'},
 {'id': 231086747, 'name': 'Подслушано у нейропсихолога', 'category': 'Школа'},
 {'id': 223898245, 'name': 'ПОДСЛУШАНО', 'category': 'Объявления'},
 {'id': 27301653, 'name': 'Подслушано ФСИН России', 'category': 'Общество'}]

In [31]:
filtered_communities_stage_1 = vk_utils.filter_communities_by_stop_words(
    communities=new_communities, state_dsn=getenv("POSTGRESQL_DSN"))
print("Отфильтровано сообществ:", len(
    new_communities) - len(filtered_communities_stage_1))
filtered_communities_stage_1[:10]

Отфильтровано сообществ: 14


[{'id': 224949751,
  'name': 'ПОДСЛУШАНО НА СВО / Выплаты, Поиск пропавших',
  'category': 'Дискуссионный клуб'},
 {'id': 185347562,
  'name': 'Подслушано ФСИН России',
  'category': 'Группа коллег'},
 {'id': 99484444,
  'name': 'Подслушано Краснодар | Типичный',
  'category': 'Городское сообщество'},
 {'id': 65140014,
  'name': 'Подслушано Родники',
  'category': 'Городское сообщество'},
 {'id': 161280734,
  'name': 'Подслушано в Назарово',
  'category': 'Городское сообщество'},
 {'id': 143277166,
  'name': 'ПОДСЛУШАНО НА СВО',
  'category': 'Дискуссионный клуб'},
 {'id': 231086747, 'name': 'Подслушано у нейропсихолога', 'category': 'Школа'},
 {'id': 223898245, 'name': 'ПОДСЛУШАНО', 'category': 'Объявления'},
 {'id': 27301653, 'name': 'Подслушано ФСИН России', 'category': 'Общество'},
 {'id': 15371203, 'name': 'Подслушано в Колпино', 'category': 'Общество'}]

In [32]:
from vk_api import VkApi
from psycopg import connect as pgsql_connect

with pgsql_connect(getenv("POSTGRESQL_DSN")) as conn:
    with conn.cursor() as cur:
        cur.execute(
            "SELECT group_id, reason_discarded "
            "FROM vk_monitor_seen "
            "WHERE reason_discarded = 1 "
            "LIMIT 10"
        )
        discarded = [row[0] for row in cur.fetchall()]

print("Отфильтрованные 1й стадией сообщества:")
api = VkApi(token=getenv("VK_API_KEY"), api_version="5.199").get_api()
discarded_names = []
if discarded:
    discarded_names = [group["name"] for group in api.groups.getById(
        group_ids=",".join(map(str, discarded)))["groups"]][:10]
discarded_names

Отфильтрованные 1й стадией сообщества:


['Подслушано Беременна в 16',
 'Подслушано про Аквариум',
 'Подслушано Border Collie',
 'Подслушано Битва Экстрасенсов 16+',
 'Подслушано в Снежном',
 'Подслушано | Платошино',
 'Подслушано Колодец',
 'Подслушано у мам',
 'Пугачевский район. Интересное',
 'Подслушано Липин Бор | Вашкинский район']

In [33]:
filtered_communities_stage_2 = vk_utils.filter_communities_semantically(api_key=getenv(
    "VK_API_KEY"), communities=filtered_communities_stage_1, state_dsn=getenv("POSTGRESQL_DSN"))
print("Отфильтровано сообществ:", len(
    filtered_communities_stage_1) - len(filtered_communities_stage_2))
filtered_communities_stage_2[:10]

Отфильтровано сообществ: 15


[{'id': 99484444,
  'name': 'Подслушано Краснодар | Типичный',
  'category': 'Городское сообщество'},
 {'id': 65140014,
  'name': 'Подслушано Родники',
  'category': 'Городское сообщество'},
 {'id': 161280734,
  'name': 'Подслушано в Назарово',
  'category': 'Городское сообщество'},
 {'id': 223898245, 'name': 'ПОДСЛУШАНО', 'category': 'Объявления'},
 {'id': 27301653, 'name': 'Подслушано ФСИН России', 'category': 'Общество'},
 {'id': 15371203, 'name': 'Подслушано в Колпино', 'category': 'Общество'},
 {'id': 163562484,
  'name': 'Подслушано Ковров',
  'category': 'Городское сообщество'},
 {'id': 225039650,
  'name': 'Подслушано Великие Луки',
  'category': 'Городское сообщество'},
 {'id': 49192814,
  'name': 'Подслушано г.Сатка',
  'category': 'Городское сообщество'},
 {'id': 70199125,
  'name': 'Подслушано Красноярск',
  'category': 'Городское сообщество'}]

In [34]:
from vk_api import VkApi
from psycopg import connect as pgsql_connect

with pgsql_connect(getenv("POSTGRESQL_DSN")) as conn:
    with conn.cursor() as cur:
        cur.execute(
            "SELECT group_id, reason_discarded "
            "FROM vk_monitor_seen "
            "WHERE reason_discarded = 2 "
            "LIMIT 10"
        )
        discarded = [row[0] for row in cur.fetchall()]

print("Отфильтрованные 2й стадией сообщества:")
api = VkApi(token=getenv("VK_API_KEY"), api_version="5.199").get_api()
discarded_names = []
if discarded:
    discarded_names = [group["name"] for group in api.groups.getById(
        group_ids=",".join(map(str, discarded)))["groups"]][:10]
discarded_names

Отфильтрованные 2й стадией сообщества:


['ПОДСЛУШАНО НА СВО / Выплаты, Поиск пропавших',
 'Подслушано ФСИН России',
 'ПОДСЛУШАНО НА СВО',
 'Подслушано у нейропсихолога',
 'Подслушано у родителей',
 'Подслушано у нейропсихолога',
 'Подслушано Ревда',
 'Подслушано у родителей школьников',
 'СЛАВЯНКА жилой район Санкт-Петербурга',
 'Ленинский район Уфы Республики Башкортостан']

In [35]:
from vk_api import VkApi

owner_id = -int("237677627")

api = VkApi(token=getenv("VK_API_KEY"), api_version="5.199").get_api()
posts = api.wall.get(owner_id=owner_id, count=10)["items"]
post_texts = [post.get("text", "") for post in posts]

post_texts

['Говорят, что изменения происходят тогда, когда становится слишком неудобно оставаться прежним.\n\nПоследние несколько месяцев стали для меня периодом большой переоценки. Я понял(а), что многие вещи, которые я считал(а) обязательными (стандарты успеха, мнение окружающих, привычка всем угождать), на самом деле просто не мои. Они навязанные, тяжелые и забирают слишком много энергии.\n\nСейчас я учусь говорить «нет» без чувства вины и «да» — своим истинным желаниям, даже если они кажутся кому-то странными или нелогичными. Это путь, и он не самый быстрый, но он мой. 👣\n\nА вы проходили через периоды «пересборки» себя? Как справлялись?\n\n#мысливслух #саморазвитие #путьксебе #перемены',
 'Пора заканчивать это безобразие! ✊\n\nМы уже сто раз говорили про мусор у железнодорожных путей, но воз и ныне там. Мусор лежит месяцами, превращая прилегающую территорию в помойку. Хватит жаловаться в пустоту, пора действовать!\n\nПредлагаю составить коллективное обращение в [название ведомства/РЖД/Админ

In [36]:
filtered_posts_stage_1 = vk_utils.filter_posts_with_faiss(posts=post_texts)
print("Отфильтровано постов:", len(post_texts) - len(filtered_posts_stage_1))
filtered_posts_stage_1[:10]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Отфильтровано постов: 5


['Пора заканчивать это безобразие! ✊\n\nМы уже сто раз говорили про мусор у железнодорожных путей, но воз и ныне там. Мусор лежит месяцами, превращая прилегающую территорию в помойку. Хватит жаловаться в пустоту, пора действовать!\n\nПредлагаю составить коллективное обращение в [название ведомства/РЖД/Администрацию]. Если будем писать по одному — нас не услышат. Пишем массово! Кто со мной? Пишите в комментариях 👇\n\n#активисты #чистыйрайон #сделаемгородчище #ждпути #вместемысила',
 'Наконец-то это свершилось! Наводим порядок! 🙌\n\nХотим выразить огромную благодарность коммунальным службам за оперативную работу! Наконец-то был полностью ликвидирован весь этот мусор у железнодорожных путей, который так долго портил нам вид.\n\nДолгое время этот мусор у железнодорожных путей был огромной проблемой для всех жителей района, но теперь на месте бывшей свалки — чистота и порядок. Мы очень рады, что этот мусор у железнодорожных путей больше не беспокоит нас и не портит экологию нашего района.\n

In [37]:
filtered_posts_stage_2 = vk_utils.filter_posts_with_llm(posts=post_texts, api=getenv(
    "OPENAI_API"), api_key=getenv("OPENAI_API_KEY"), model=getenv("OPENAI_API_MODEL"))
print("Отфильтровано постов:", len(
    filtered_posts_stage_1) - len(filtered_posts_stage_2))
filtered_posts_stage_2[:10]

Отфильтровано постов: 1


['Пора заканчивать это безобразие! ✊\n\nМы уже сто раз говорили про мусор у железнодорожных путей, но воз и ныне там. Мусор лежит месяцами, превращая прилегающую территорию в помойку. Хватит жаловаться в пустоту, пора действовать!\n\nПредлагаю составить коллективное обращение в [название ведомства/РЖД/Администрацию]. Если будем писать по одному — нас не услышат. Пишем массово! Кто со мной? Пишите в комментариях 👇\n\n#активисты #чистыйрайон #сделаемгородчище #ждпути #вместемысила',
 'Соседи, вы заметили, как сильно изменилась ситуация с мусором у путей? ⚠\n\nЭто уже не просто вопрос красоты. Эти залежи мусора у железной дороги — это рассадник инфекции и грызунов. С ветром этот запах разносится по всем ближайшим домам. К тому же, если это разлетится или кто-то решит развести там костер, последствия будут плачевными.\n\nМы не можем просто закрывать на это глаза. Нужно коллективное обращение в администрацию и к перевозчикам. Кто готов подписаться?\n\n#экология #безопасность #здоровье #сосе